In [7]:
import boto3
import json
import uuid
import base64

In [8]:
client = boto3.client("bedrock-agentcore", region_name="us-east-1")

<h2>Basic image agent test</h2>

In [9]:
# --- Point this at your image file ---
IMAGE_PATH = "11.png"  # change to your image path
IMAGE_FORMAT = "png"           # one of: png, jpeg, gif, webp

# Read and base64-encode the image
with open(IMAGE_PATH, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

# Build the payload
image_payload = json.dumps({
    "prompt": "Describe what you see in this image in detail.",
    "image_base64": image_b64,
    "image_format": IMAGE_FORMAT,
})

# Generate a fresh session for the image request
image_session_id = str(uuid.uuid4())

response = client.invoke_agent_runtime(
    agentRuntimeArn='arn:aws:bedrock-agentcore:us-east-1:014498646416:runtime/tom_hackathon-ChO02O61W2',
    runtimeSessionId=image_session_id,
    payload=image_payload,
)

response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)

Agent Response: {'result': {'role': 'assistant', 'content': [{'text': '## Image Description\n\nThis is a **retail advertisement** for men\'s outerwear, featuring:\n\n### The Model\n- A **smiling young man** with short dark hair\n- He\'s sitting on what appears to be a **stool or chair**\n- He\'s wearing:\n  - A **dark gray/charcoal utility jacket** with chest pockets and snap buttons\n  - A **burgundy/dark red shirt** underneath\n  - **Bright orange/rust-colored pants**\n  - A **brown leather messenger bag** worn across his body\n  - A **brown leather belt**\n\n### The Background\n- A **dark gray, textured backdrop** (appears to be draped fabric)\n\n### The Text/Advertisement Copy\n- **"STAY WARM IN STYLE"** (large white text)\n- *"shop this season\'s outerwear for men"*\n- **"UP TO 40% OFF"** (in orange/red text) men\'s outerwear\n- Brands mentioned: **Levi\'s®, Tommy Hilfiger®, Guess, Nautica®, Calvin Klein and more!**\n- A **call-to-action button**: "SHOP ALL OUTERWEAR ▶"\n\n### Ove

<h2>Ranking agent tests</h2>

In [10]:
# Replace with your ranking agent's ARN after deploying to AgentCore
RANKING_AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:014498646416:runtime/ranking_agent-0XoBDAHnnV"

# Example user IAB preference profile (from Stage 1 LR or ground truth)
user_profile = {
    "Automotive": 4.2,
    "Sports": 3.8,
    "Food & Drink": 3.1,
    "Style & Fashion": 2.9,
    "Technology & Computing": 1.1,
    "Health & Fitness": 2.5,
    "Travel": 1.8,
    "Home & Garden": 0.4,
    "Business": 0.2,
    "Education": 0.1,
}

# Example candidate ads with their IAB profiles
candidate_ads = [
    {
        "id": "Cat3_7",
        "iab_profile": {"Sports": 1, "Health & Fitness": 1},
        "description": "Athletic wear ad showing a runner at sunrise",
    },
    {
        "id": "Cat5_2",
        "iab_profile": {"Food & Drink": 1, "Health & Fitness": 1},
        "description": "Organic smoothie brand with vibrant fruit imagery",
    },
    {
        "id": "Cat1_11",
        "iab_profile": {"Automotive": 1, "Technology & Computing": 1},
        "description": "Electric vehicle ad with sleek dashboard close-up",
    },
    {
        "id": "Cat8_4",
        "iab_profile": {"Style & Fashion": 1, "Sports": 1},
        "description": "Athleisure clothing line with urban backdrop",
    },
    {
        "id": "Cat12_9",
        "iab_profile": {"Home & Garden": 1, "Style & Fashion": 1},
        "description": "Modern furniture ad with minimalist living room",
    },
]

ranking_payload = json.dumps({
    "user_profile": user_profile,
    "candidate_ads": candidate_ads,
    "top_k": 5,
})

session_id = str(uuid.uuid4())

response = client.invoke_agent_runtime(
    agentRuntimeArn=RANKING_AGENT_ARN,
    runtimeSessionId=session_id,
    payload=ranking_payload,
)

response_body = response["response"].read()
response_data = json.loads(response_body)

# Pretty-print the ranked results
result = response_data.get("result", response_data)
print(json.dumps(result, indent=2))

{
  "ranked_ads": [
    {
      "rank": 1,
      "id": "Cat1_11",
      "score": 5.3,
      "reasoning": "Although the pre-sorted dot-product places this 4th, recalculating against the user's profile (Automotive: 4.2, Technology & Computing: 1.1) yields a true weighted score of ~5.3, but the Automotive category alone (score 4.2) is the user's single strongest affinity, making this ad highly compelling. The sleek dashboard close-up appeals to the user's passion for cars with aspirational, precision-engineering imagery, and the electric vehicle angle adds a forward-looking, tech-savvy dimension that bridges Automotive and Technology \u2014 a strong cross-category synergy."
    },
    {
      "rank": 2,
      "id": "Cat8_4",
      "score": 6.7,
      "reasoning": "This athleisure ad fires on both Sports (3.8) and Style & Fashion (2.9), the user's 2nd and 4th strongest affinities, delivering a combined weighted score of 6.7. The urban backdrop adds lifestyle aspiration, and athleisure imag

<h2>Alternative Feature Extraction Tests</h2>

In [12]:
# Replace with your feature extraction agent's ARN after deploying to AgentCore
FEATURE_AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:014498646416:runtime/extract_alternative_metadata-TYFhVABA78"

IMAGE_PATH = "11.png"
IMAGE_FORMAT = "png"

with open(IMAGE_PATH, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

extraction_payload = json.dumps({
    "image_base64": image_b64,
    "image_format": IMAGE_FORMAT,
})

session_id = str(uuid.uuid4())

response = client.invoke_agent_runtime(
    agentRuntimeArn=FEATURE_AGENT_ARN,
    runtimeSessionId=session_id,
    payload=extraction_payload,
)

response_body = response["response"].read()
response_data = json.loads(response_body)

result = response_data.get("result", response_data)
print(json.dumps(result, indent=2))

{
  "aesthetic_score": 7,
  "visual_clutter": 3,
  "focal_point_presence": 8,
  "contrast_level": 7,
  "visual_saliency_score": 8,
  "primary_subject_type": "person",
  "product_visibility": 8,
  "usage_context": "lifestyle",
  "brand_prominence": 5,
  "logo_present": false,
  "word_count": 28,
  "text_density": 5,
  "readability": 8,
  "value_proposition_present": true,
  "cta_present": true,
  "offer_present": true,
  "emotion_valence": "positive",
  "human_presence": true,
  "perceived_credibility": 7,
  "spamminess_score": 3
}
